In [ ]:
import os
import sys
import talib
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
%matplotlib inline
sns.set_theme()

In [ ]:
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

In [ ]:
from src.Alpha9.utility import get_config, read_file

In [ ]:
# read data for a specific crypto
i = 0
config = get_config.load()
symbols = config['pipeline']['symbols']
data = read_file.read_data('raw', symbols[i])

In [ ]:
data['rsi'] = talib.RSI(data['close'], timeperiod=14)
data['atr'] = talib.ATR(data['high'], data['low'], data['close'], timeperiod=14)
data['adx'] = talib.ADX(data['high'], data['low'], data['close'], timeperiod=14)
data['sma50'] = talib.SMA(data['close'], timeperiod=50)

In [ ]:
def normalize(data, columns):
    for col in columns:
        window = data[col].rolling(window=30, min_periods=30)
        data[col] = (data[col] - window.mean()) / window.std()

In [ ]:
normalize(data, list(data.columns))
data.dropna(inplace=True)

In [ ]:
def plot_heatmap(corr_matrix):
    plt.figure(figsize=(16, 12))
    sns.heatmap(
        corr_matrix,
        annot=True,
        fmt=".2f",
        cmap='coolwarm',
        vmin=-1, vmax=1,
        linewidths=0.5,
        cbar_kws={"shrink": 0.8}
    )

    plt.title("Correlation Matrix - Features", fontsize=16, color='blue')
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

In [ ]:
def evaluate_feature(data, features):
    results = []

    # plot correlation heatmap
    plot_heatmap(data[features].corr())

    # calculate metrics
    for feature in features:
        result = {'Feature' : feature}

        # calculate redundancy score
        corrs = data[features].corrwith(data[feature])
        others = corrs.drop(feature)
        result['Redundancy Score'] = np.linalg.norm(others)

        # calculate stationarity
        adf = adfuller(data[feature].values)
        result['Stationarity Value'] = f'{adf[1]:.6e}'

        results.append(result)

    return results

In [ ]:
result = evaluate_feature(data, ['rsi', 'atr', 'adx', 'sma50'])
print(pd.DataFrame(result))